# 03 — Nouveautés Python 3.11 a 3.14

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- utiliser les `ExceptionGroup` et `except*` (PEP 654, Python 3.11) ;
- comprendre `Self` et `LiteralString` (PEP 673, Python 3.11) ;
- utiliser la syntaxe `type X = ...` et les generics `[T]` (PEP 695, Python 3.12) ;
- écrire des f-strings imbriquées (PEP 701, Python 3.12) ;
- comprendre les enjeux du free-threading (PEP 703, Python 3.13) ;
- utiliser `from __future__ import annotations` et PEP 649 (Python 3.14) ;
- découvrir les template strings `t"..."` (PEP 750, Python 3.14).

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- les exceptions, le modèle objet, les type hints ;
- les generics avec `TypeVar` (ancienne syntaxe) ;
- les f-strings et le formatage ;
- `asyncio` et la concurrence (notebooks précédents).

## Plan

1. Python 3.11 — ExceptionGroup (PEP 654)
2. Python 3.11 — `Self` et `LiteralString` (PEP 673)
3. Python 3.11 — Autres nouveautés
4. Python 3.12 — Syntaxe `type` et generics (PEP 695)
5. Python 3.12 — f-strings améliorées (PEP 701)
6. Python 3.12 — Autres nouveautés
7. Python 3.13 — Free-threading expérimental (PEP 703)
8. Python 3.13 — Autres nouveautés
9. Python 3.14 — Annotations lazy (PEP 649)
10. Python 3.14 — Template strings (PEP 750)
11. Python 3.14 — Autres nouveautés
12. Synthèse
13. Exercices
14. Ressources

---

## 1. Python 3.11 — ExceptionGroup (PEP 654)

Les `ExceptionGroup` permettent de lever et capturer **plusieurs exceptions simultanément**. C'est essentiel pour `asyncio.TaskGroup` et les bibliothèques de concurrence.

In [ ]:
# Créer un ExceptionGroup
eg = ExceptionGroup("erreurs multiples", [
    ValueError("valeur incorrecte"),
    TypeError("type incorrect"),
    OSError("fichier introuvable"),
])
print(eg)
print(f"Nombre d'exceptions : {len(eg.exceptions)}")

### `except*` — capturer par type

In [ ]:
try:
    raise ExceptionGroup("erreurs", [
        ValueError("val1"),
        ValueError("val2"),
        TypeError("type1"),
    ])
except* ValueError as eg:
    print(f"ValueError capturées : {eg.exceptions}")
except* TypeError as eg:
    print(f"TypeError capturées : {eg.exceptions}")

**Attention :** `except*` n'est pas un `except` classique. Il peut capturer **un sous-ensemble** des exceptions du groupe, et les autres continuent de se propager.

### Utilisation avec `asyncio.TaskGroup`

In [ ]:
import asyncio

async def tache_ok():
    return 42

async def tache_erreur(msg):
    raise ValueError(msg)

async def main():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(tache_ok())
            tg.create_task(tache_erreur("erreur A"))
            tg.create_task(tache_erreur("erreur B"))
    except* ValueError as eg:
        print(f"Erreurs capturées : {[str(e) for e in eg.exceptions]}")

asyncio.run(main())

### Filtrer les exceptions dans un groupe

In [ ]:
def filtrer_group(eg, type_):
    """Retourne les exceptions d'un type donné dans un ExceptionGroup."""
    return [e for e in eg.exceptions if isinstance(e, type_)]

eg = ExceptionGroup("test", [
    ValueError("v1"),
    TypeError("t1"),
    ValueError("v2"),
    OSError("o1"),
])

print(f"ValueError : {filtrer_group(eg, ValueError)}")
print(f"OSError    : {filtrer_group(eg, OSError)}")

### `subgroup()` et `split()`

In [ ]:
eg = ExceptionGroup("mix", [
    ValueError("v1"),
    TypeError("t1"),
    ValueError("v2"),
])

# subgroup : garde uniquement les exceptions d'un type
sub = eg.subgroup(ValueError)
print(f"subgroup ValueError : {sub}")

# split : sépare en deux groupes
match, rest = eg.split(ValueError)
print(f"match : {match}")
print(f"rest  : {rest}")

---

## 2. Python 3.11 — `Self` et `LiteralString` (PEP 673)

`Self` permet d'annoter les méthodes qui retournent l'instance courante, même dans les sous-classes.

In [ ]:
from typing import Self

class Builder:
    def __init__(self):
        self.options: dict = {}

    def option(self, key: str, value: str) -> Self:
        self.options[key] = value
        return self  # Self = le type réel de l'instance

    def build(self) -> dict:
        return self.options.copy()

result = Builder().option("a", "1").option("b", "2").build()
print(result)

Sans `Self`, il fallait utiliser `TypeVar` avec `bound=...`, ce qui était verbeux et error-prone.

### `LiteralString` — sécurité contre l'injection

In [ ]:
from typing import LiteralString

def executer_sql(query: LiteralString) -> None:
    """N'accepte que des chaînes littérales (pas des inputs utilisateur)."""
    print(f"SQL : {query}")

# OK — chaîne littérale
executer_sql("SELECT * FROM users WHERE id = ?")

# mypy refuserait ceci :
# user_input = input("query: ")
# executer_sql(user_input)  # erreur mypy : pas un LiteralString

---

## 3. Python 3.11 — Autres nouveautés

### Messages d'erreur améliorés

In [ ]:
# Python 3.11 indique EXACTEMENT quelle expression a causé l'erreur
# avec des ^^^^^^ sous la ligne fautive

# Exemple (le traceback est plus précis) :
try:
    data = {"a": {"b": None}}
    result = data["a"]["b"]["c"]  # TypeError plus précis
except TypeError as e:
    print(f"Erreur : {e}")

### `tomllib` — lecture TOML dans la stdlib

In [ ]:
import tomllib

toml_str = b'''
[project]
name = "mon-app"
version = "1.0.0"

[project.dependencies]
requests = ">=2.28"
'''

config = tomllib.loads(toml_str.decode())
print(f"Nom : {config['project']['name']}")
print(f"Version : {config['project']['version']}")

### Performances : CPython 3.11 est 10-60 % plus rapide

Le projet **Faster CPython** (Shannon plan) a introduit :
- **Specializing Adaptive Interpreter** : les opcodes se spécialisent au runtime ;
- **Lazy frame creation** : les frames ne sont créées que si nécessaire ;
- **Zero-cost exceptions** : les blocs `try` ne coûtent rien si aucune exception n'est levée.

---

## 4. Python 3.12 — Syntaxe `type` et generics (PEP 695)

La PEP 695 introduit une nouvelle syntaxe pour les generics et les alias de types, beaucoup plus lisible que l'ancienne avec `TypeVar`.

### Ancienne syntaxe (avant 3.12)

In [ ]:
from typing import TypeVar

T_old = TypeVar("T_old")

def premier_old(lst: list[T_old]) -> T_old:
    return lst[0]

print(premier_old([1, 2, 3]))
print(premier_old(["a", "b"]))

### Nouvelle syntaxe (Python 3.12+)

In [ ]:
# Syntaxe PEP 695 — beaucoup plus lisible
def premier[T](lst: list[T]) -> T:
    return lst[0]

print(premier([1, 2, 3]))
print(premier(["a", "b"]))

### Classes generiques

In [ ]:
class Pile[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def __repr__(self) -> str:
        return f"Pile({self._items})"

p = Pile[int]()
p.push(1)
p.push(2)
print(p)
print(p.pop())

### `type` statement — alias de types

In [ ]:
# Ancien style
# Matrix = list[list[float]]

# Nouveau style (PEP 695)
type Matrix = list[list[float]]
type Point = tuple[float, float]
type Callback[T] = Callable[[T], None]

from typing import Callable

m: Matrix = [[1.0, 2.0], [3.0, 4.0]]
print(f"Matrix : {m}")

### Generics bornés et contraints

In [ ]:
from typing import SupportsFloat

# T borné (upper bound)
def double[T: SupportsFloat](x: T) -> float:
    return float(x) * 2

print(double(3))
print(double(3.14))

---

## 5. Python 3.12 — f-strings améliorées (PEP 701)

Avant Python 3.12, les f-strings avaient des limitations : pas de backslash, pas d'imbrication, pas de commentaires. La PEP 701 les lève toutes.

### Imbrication de f-strings

In [ ]:
# Avant 3.12 : impossible
# Depuis 3.12 : parfaitement légal
data = {"name": "Alice", "role": "admin"}
print(f"Utilisateur : {f'{data["name"]} ({data["role"]})'}")

### Backslash dans les f-strings

In [ ]:
# Avant 3.12 : SyntaxError
# Depuis 3.12 : autorisé
items = ["a", "b", "c"]
print(f"Items : {chr(10).join(items)}")

### Expressions multi-lignes

In [ ]:
# Les f-strings peuvent contenir des expressions complexes
result = f"{
    sum(
        x ** 2
        for x in range(10)
    )
}"
print(f"Somme des carrés : {result}")

---

## 6. Python 3.12 — Autres nouveautés

### PEP 709 — Inline comprehensions

Les compréhensions ne créent plus de frame séparée. Elles sont compilées **inline** dans le bytecode de la fonction englobante. Cela les rend plus rapides (~2x pour les petites compréhensions).

### `itertools.batched()`

In [ ]:
from itertools import batched

# Découper un itérable en lots de n éléments
data = range(10)
for lot in batched(data, 3):
    print(lot)

### `pathlib.Path.walk()`

In [ ]:
from pathlib import Path
import tempfile
import os

# Path.walk() — équivalent de os.walk() pour pathlib
with tempfile.TemporaryDirectory() as d:
    (Path(d) / "sub").mkdir()
    (Path(d) / "file.txt").touch()
    (Path(d) / "sub" / "inner.txt").touch()

    for root, dirs, files in Path(d).walk():
        for f in files:
            print(root / f)

---

## 7. Python 3.13 — Free-threading expérimental (PEP 703)

La PEP 703 propose de rendre le **GIL optionnel** dans CPython. Python 3.13 inclut une version **expérimentale** de CPython sans GIL (free-threaded).

### Le problème du GIL

| Aspect | Avec GIL | Sans GIL (free-threaded) |
|---|---|---|
| Threads Python | Un seul exécute du Python à la fois | Vrai parallélisme |
| Code CPU-bound | Pas de gain avec les threads | Gain proportionnel aux cores |
| Code I/O-bound | Fonctionne bien | Fonctionne bien |
| Extensions C | Peuvent relâcher le GIL | Doivent être thread-safe |
| Compatibilité | 100 % | Extensions C à adapter |

### Installation

```bash
# Build free-threaded (suffixe 't')
python3.13t --version

# Ou via pyenv
pyenv install 3.13.0t
```

### Vérifier si le GIL est actif

In [ ]:
import sys

if hasattr(sys, "_is_gil_enabled"):
    print(f"GIL activé : {sys._is_gil_enabled()}")
else:
    print("API _is_gil_enabled non disponible (Python < 3.13 ou build standard)")

### Implications pratiques

- **Ne migrez pas en production** : c'est expérimental en 3.13.
- **Testez vos extensions C** : elles doivent être thread-safe.
- **`numpy`, `pandas`** : travaillent sur la compatibilité.
- Le free-threading sera la **norme** à terme (objectif : Python 3.17+).

---

## 8. Python 3.13 — Autres nouveautés

### REPL amélioré

Le REPL de Python 3.13 est entièrement réécrit :
- Coloration syntaxique ;
- Édition multi-lignes ;
- Historique amélioré ;
- Collage de blocs de code.

### `dbm.sqlite3` — nouveau backend par défaut

In [ ]:
# dbm utilise maintenant SQLite par défaut
import dbm
print(f"Backend dbm par défaut : {dbm.whichdb.__module__ if hasattr(dbm, 'whichdb') else 'sqlite3 (3.13+)'}")

### Messages de TypeError améliorés

In [ ]:
# Python 3.13 donne des messages d'erreur encore plus précis
try:
    "hello" + 42
except TypeError as e:
    print(f"Erreur : {e}")

---

## 9. Python 3.14 — Annotations lazy (PEP 649)

La PEP 649 change fondamentalement le moment où les annotations sont évaluées. Au lieu d'être évaluées **à la définition** de la classe/fonction, elles sont évaluées **à la demande** (quand on accède à `__annotations__`).

### Le problème avant PEP 649

```python
# Avant : les annotations sont évaluées immédiatement
class Noeud:
    def enfants(self) -> list[Noeud]:  # NameError : Noeud pas encore défini !
        ...
```

Solutions de contournement :
- `from __future__ import annotations` (PEP 563) — stocke les annotations comme chaînes ;
- Forward reference : `"Noeud"` entre guillemets.

### Avec PEP 649 (Python 3.14+)

Les annotations sont stockées comme du **code exécutable** (pas des chaînes) mais évaluées **paresseusement**. Plus besoin de `from __future__ import annotations`.

In [ ]:
# Python 3.14 : ceci fonctionne nativement
class Noeud:
    def __init__(self, valeur: int) -> None:
        self.valeur = valeur
        self.enfants: list[Noeud] = []  # Pas d'erreur !

    def ajouter(self, enfant: Noeud) -> None:
        self.enfants.append(enfant)

n = Noeud(1)
n.ajouter(Noeud(2))
print(f"Enfants : {[e.valeur for e in n.enfants]}")

### `annotationlib` — nouveau module

In [ ]:
# Python 3.14 ajoute annotationlib pour inspecter les annotations
try:
    import annotationlib
    print("annotationlib disponible")
except ImportError:
    print("annotationlib non disponible (Python < 3.14)")

---

## 10. Python 3.14 — Template strings (PEP 750)

La PEP 750 introduit les **template strings** avec le préfixe `t"..."`. Contrairement aux f-strings qui produisent une `str`, les t-strings produisent un objet `Template` qui peut être inspecté et transformé **avant** d'être converti en chaîne.

### Pourquoi les template strings ?

Les f-strings sont puissantes mais dangereuses pour :
- **SQL** : `f"SELECT * FROM users WHERE name = '{name}'"` → injection SQL ;
- **HTML** : `f"<p>{user_input}</p>"` → XSS ;
- **Logging** : on veut les données structurées, pas une chaîne aplatie.

Les template strings permettent d'**intercepter les valeurs** avant interpolation.

In [ ]:
# Python 3.14+
import sys
if sys.version_info >= (3, 14):
    # t-strings retournent un objet Template
    nom = "Alice"
    # template = t"Bonjour {nom}"
    # print(type(template))  # <class 'string.templatelib.Template'>
    # print(template.args)   # les morceaux statiques + dynamiques
    print("Template strings disponibles en Python 3.14+")
else:
    print(f"Python {sys.version_info[:2]} — template strings non disponibles")
    print("Syntaxe : t'Bonjour {nom}' retourne un objet Template")

### Cas d'usage : requête SQL sûre

```python
# Avec t-strings (Python 3.14)
from string.templatelib import Template

def sql(template: Template) -> tuple[str, list]:
    query_parts = []
    params = []
    for part in template.args:
        if isinstance(part, str):
            query_parts.append(part)
        else:
            query_parts.append("?")
            params.append(part.value)
    return "".join(query_parts), params

name = "Alice"
query, params = sql(t"SELECT * FROM users WHERE name = {name}")
# query = "SELECT * FROM users WHERE name = ?"
# params = ["Alice"]
```

### Cas d'usage : HTML sûr

```python
def html(template: Template) -> str:
    parts = []
    for part in template.args:
        if isinstance(part, str):
            parts.append(part)
        else:
            # Échapper automatiquement les valeurs
            from html import escape
            parts.append(escape(str(part.value)))
    return "".join(parts)

user_input = '<script>alert("xss")</script>'
safe = html(t"<p>{user_input}</p>")
# <p>&lt;script&gt;alert(&quot;xss&quot;)&lt;/script&gt;</p>
```

---

## 11. Python 3.14 — Autres nouveautés

### Performances continues

Chaque version de Python est plus rapide que la précédente grâce au projet Faster CPython :

| Version | Gain vs 3.10 |
|---|---|
| 3.11 | +10-60 % |
| 3.12 | +5 % supplémentaires |
| 3.13 | +5 % (JIT expérimental) |
| 3.14 | En cours de mesure |

### JIT compiler (expérimental)

Python 3.13+ inclut un **JIT compiler expérimental** (copy-and-patch JIT). Il n'est pas activé par défaut mais montre des gains prometteurs sur les boucles critiques.

### Deprecations et retraits

| Retiré/Déprécié | Version | Remplacement |
|---|---|---|
| `imp` module | 3.12 (retiré) | `importlib` |
| `distutils` | 3.12 (retiré) | `setuptools` |
| `asyncio.coroutine` | 3.11 (retiré) | `async def` |
| `typing.TypeAlias` | 3.12 (déprécié) | `type X = ...` |
| `from __future__ import annotations` | 3.14 (obsolète) | PEP 649 natif |

---

## 12. Synthèse

| PEP | Version | Fonctionnalité |
|---|---|---|
| **654** | 3.11 | `ExceptionGroup` et `except*` |
| **673** | 3.11 | `Self`, `LiteralString` |
| **695** | 3.12 | `type X = ...`, `def f[T](...)` |
| **701** | 3.12 | f-strings imbriquées, backslash autorisé |
| **709** | 3.12 | Compréhensions inline (perf) |
| **703** | 3.13 | Free-threading (GIL optionnel, expérimental) |
| **649** | 3.14 | Annotations lazy (plus besoin de `__future__`) |
| **750** | 3.14 | Template strings `t"..."` |

**Règles à retenir :**
- `except*` est indispensable pour `asyncio.TaskGroup`.
- La syntaxe `def f[T](...)` remplace `TypeVar` — adoptez-la dès Python 3.12.
- Le free-threading est l'avenir, mais pas encore pour la production.
- Les template strings résolvent le problème des injections SQL/HTML.
- Mettez à jour Python régulièrement : chaque version apporte des gains de performance.

---

## 13. Exercices

### Exercice 1 — ExceptionGroup handler *(facile)*

Écrire une fonction `run_all(functions)` qui exécute toutes les fonctions de la liste. Si certaines lèvent des exceptions, les collecter dans un `ExceptionGroup` et le lever à la fin.

```python
def ok(): return 42
def err1(): raise ValueError("v1")
def err2(): raise TypeError("t1")

run_all([ok, err1, ok, err2])
# → ExceptionGroup avec ValueError et TypeError
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Nouveautes_3_11_a_3_14", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
def run_all(functions):
    erreurs = []
    resultats = []
    for fn in functions:
        try:
            resultats.append(fn())
        except Exception as e:
            erreurs.append(e)
    if erreurs:
        raise ExceptionGroup(f"{len(erreurs)} erreur(s)", erreurs)
    return resultats

def ok(): return 42
def err1(): raise ValueError("v1")
def err2(): raise TypeError("t1")

try:
    run_all([ok, err1, ok, err2])
except* ValueError as eg:
    print(f"ValueError : {eg.exceptions}")
except* TypeError as eg:
    print(f"TypeError : {eg.exceptions}")
```

</details>

### Exercice 2 — Classe generique PEP 695 *(facile)*

Réécrire la classe suivante avec la syntaxe PEP 695 (Python 3.12+) :

```python
from typing import TypeVar, Generic
T = TypeVar("T")

class File(Generic[T]):
    def __init__(self) -> None:
        self._items: list[T] = []
    def enfiler(self, item: T) -> None:
        self._items.append(item)
    def defiler(self) -> T:
        return self._items.pop(0)
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Nouveautes_3_11_a_3_14", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
class File[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def enfiler(self, item: T) -> None:
        self._items.append(item)

    def defiler(self) -> T:
        return self._items.pop(0)

    def __repr__(self) -> str:
        return f"File({self._items})"

f = File[int]()
f.enfiler(1)
f.enfiler(2)
print(f.defiler())
print(f)
```

</details>

### Exercice 3 — Builder pattern avec `Self` *(moyen)*

Implémenter un `QueryBuilder` avec chaînage fluide, en utilisant `Self` pour que les sous-classes conservent le bon type de retour.

```python
class QueryBuilder:
    def select(self, *cols) -> Self: ...
    def where(self, condition) -> Self: ...
    def limit(self, n) -> Self: ...
    def build(self) -> str: ...
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Nouveautes_3_11_a_3_14", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Self

class QueryBuilder:
    def __init__(self, table: str) -> None:
        self._table = table
        self._columns: list[str] = ["*"]
        self._conditions: list[str] = []
        self._limit: int | None = None

    def select(self, *cols: str) -> Self:
        self._columns = list(cols)
        return self

    def where(self, condition: str) -> Self:
        self._conditions.append(condition)
        return self

    def limit(self, n: int) -> Self:
        self._limit = n
        return self

    def build(self) -> str:
        query = f"SELECT {', '.join(self._columns)} FROM {self._table}"
        if self._conditions:
            query += " WHERE " + " AND ".join(self._conditions)
        if self._limit is not None:
            query += f" LIMIT {self._limit}"
        return query

query = (
    QueryBuilder("users")
    .select("name", "email")
    .where("age > 18")
    .where("active = true")
    .limit(10)
    .build()
)
print(query)
```

</details>

### Exercice 4 — Retry avec ExceptionGroup *(moyen)*

Écrire une fonction `retry_all(tasks, max_retries=3)` qui :
1. Exécute toutes les tâches ;
2. Celles qui échouent sont réessayées (jusqu'à `max_retries` fois) ;
3. Les erreurs finales sont collectées dans un `ExceptionGroup`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Nouveautes_3_11_a_3_14", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import random

def retry_all(tasks, max_retries=3):
    resultats = {}
    erreurs_finales = []

    for i, task in enumerate(tasks):
        for attempt in range(max_retries):
            try:
                resultats[i] = task()
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    erreurs_finales.append(e)

    if erreurs_finales:
        raise ExceptionGroup(
            f"{len(erreurs_finales)} tâche(s) en échec après {max_retries} tentatives",
            erreurs_finales,
        )
    return resultats

# Tâches avec échec aléatoire
def tache_aleatoire():
    if random.random() < 0.7:
        raise ValueError("échec aléatoire")
    return "ok"

try:
    results = retry_all([tache_aleatoire] * 5, max_retries=5)
    print(f"Tout réussi : {results}")
except* ValueError as eg:
    print(f"{len(eg.exceptions)} tâches en échec final")
```

</details>

### Exercice 5 — Migration de types anciens vers PEP 695 *(difficile)*

Réécrire le code suivant en utilisant exclusivement la syntaxe PEP 695 :

```python
from typing import TypeVar, Generic, Protocol, runtime_checkable

T = TypeVar("T")
K = TypeVar("K")
V = TypeVar("V")
Comparable = TypeVar("Comparable", bound="SupportsLessThan")

@runtime_checkable
class SupportsLessThan(Protocol):
    def __lt__(self, other: "SupportsLessThan") -> bool: ...

class SortedDict(Generic[K, V]):
    def __init__(self) -> None:
        self._data: dict[K, V] = {}
    def set(self, key: K, value: V) -> None:
        self._data[key] = value
    def keys_sorted(self) -> list[K]:
        return sorted(self._data.keys())
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Nouveautes_3_11_a_3_14", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class SupportsLessThan(Protocol):
    def __lt__(self, other: "SupportsLessThan") -> bool: ...

class SortedDict[K: SupportsLessThan, V]:
    def __init__(self) -> None:
        self._data: dict[K, V] = {}

    def set(self, key: K, value: V) -> None:
        self._data[key] = value

    def keys_sorted(self) -> list[K]:
        return sorted(self._data.keys())

    def __repr__(self) -> str:
        return f"SortedDict({self._data})"

sd = SortedDict[str, int]()
sd.set("banane", 3)
sd.set("pomme", 1)
sd.set("abricot", 2)
print(f"Clés triées : {sd.keys_sorted()}")
```

</details>

---

## 14. Ressources

- [What's New in Python 3.11](https://docs.python.org/3/whatsnew/3.11.html)
- [What's New in Python 3.12](https://docs.python.org/3/whatsnew/3.12.html)
- [What's New in Python 3.13](https://docs.python.org/3/whatsnew/3.13.html)
- [PEP 654 — Exception Groups](https://peps.python.org/pep-0654/)
- [PEP 695 — Type Parameter Syntax](https://peps.python.org/pep-0695/)
- [PEP 701 — Syntactic formalization of f-strings](https://peps.python.org/pep-0701/)
- [PEP 703 — Making the GIL Optional](https://peps.python.org/pep-0703/)
- [PEP 649 — Deferred Evaluation of Annotations](https://peps.python.org/pep-0649/)
- [PEP 750 — Template Strings](https://peps.python.org/pep-0750/)
- [Faster CPython](https://github.com/faster-cpython/ideas)